# audiogram-object: Object Model Intro

Walkthrough of `ThresholdPoint`, `EarAudiogram`, and `BinauralAudiogram`.

Covers construction, metrics, and asymmetry.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from audiogram_object import (
    ThresholdPoint,
    EarAudiogram,
    BinauralAudiogram,
    ASYMMETRY_CRITERIA,
)
from pprint import pprint

## ThresholdPoint

The atomic unit — one frequency, one measurement.
- `threshold_db`: the measured level (or max output tested if NR)
- `masked`: was the non-test ear masked?
- `nr`: no response at this frequency

In [3]:
# Plain threshold — defaults masked=False, nr=False
pt = ThresholdPoint(25.0)
print(pt)

# Masked
pt_masked = ThresholdPoint(30.0, masked=True)
print(pt_masked)

# No response — store max output level with nr=True
pt_nr = ThresholdPoint(120.0, nr=True)
print(pt_nr)

ThresholdPoint(threshold_db=25.0, masked=False, nr=False)
ThresholdPoint(threshold_db=30.0, masked=True, nr=False)
ThresholdPoint(threshold_db=120.0, masked=False, nr=True)


## EarAudiogram

One ear's thresholds — air and bone conduction.

Accepts plain floats (coerced to `ThresholdPoint`) or full `ThresholdPoint` objects.

In [4]:
# Float shorthand — fine for simple cases
ear_simple = EarAudiogram(air={500: 20.0, 1000: 25.0, 2000: 30.0, 4000: 45.0})
print('air[500]:', ear_simple.air[500])  # auto-coerced to ThresholdPoint

air[500]: ThresholdPoint(threshold_db=20.0, masked=False, nr=False)


In [5]:
# Full clinical detail — air + bone, masked bone, NR at 4k
left = EarAudiogram(
    air={
        250:  ThresholdPoint(10.0),
        500:  ThresholdPoint(15.0),
        1000: ThresholdPoint(30.0),
        2000: ThresholdPoint(40.0),
        4000: ThresholdPoint(50.0),
        8000: ThresholdPoint(65.0),
    },
    bone={
        500:  ThresholdPoint(10.0, masked=True),
        1000: ThresholdPoint(20.0, masked=True),
        2000: ThresholdPoint(30.0, masked=True),
        4000: ThresholdPoint(45.0, masked=True),
    },
)

right = EarAudiogram(
    air={
        250:  ThresholdPoint(15.0),
        500:  ThresholdPoint(40.0),
        1000: ThresholdPoint(55.0),
        2000: ThresholdPoint(60.0),
        4000: ThresholdPoint(120.0, nr=True),
        8000: ThresholdPoint(120.0, nr=True),
    },
    bone={
        500:  ThresholdPoint(35.0, masked=True),
        1000: ThresholdPoint(50.0, masked=True),
        2000: ThresholdPoint(55.0, masked=True),
    },
)

print('Left air frequencies:', left.available_frequencies())
print('Left bone frequencies:', left.available_frequencies('bone'))
print('Right air[4000]:', right.air[4000])

Left air frequencies: [250, 500, 1000, 2000, 4000, 8000]
Left bone frequencies: [500, 1000, 2000, 4000]
Right air[4000]: ThresholdPoint(threshold_db=120.0, masked=False, nr=True)


## Metrics — PTA

In [6]:
# Standard 3-frequency PTA (500, 1000, 2000 Hz)
print('Left PTA (air):', left.pta())
print('Right PTA (air):', right.pta())

# 4-frequency PTA
print('Left 4-freq PTA:', left.pta(freqs=(500, 1000, 2000, 4000)))

# Bone PTA
print('Left PTA (bone, masked):', left.pta(pathway='bone'))

# require_all — returns None if any requested freq is missing
print('Right PTA (require_all=True):', right.pta(freqs=(500, 1000, 2000), require_all=True))

Left PTA (air): 28.333333333333332
Right PTA (air): 51.666666666666664
Left 4-freq PTA: 33.75
Left PTA (bone, masked): 20.0
Right PTA (require_all=True): 51.666666666666664


## Metrics — Air-Bone Gap & Loss Type

ABG is computed per-frequency where both air and bone thresholds exist.

ABG PTA averages the gaps. Loss type classifies based on air PTA, bone PTA, and ABG PTA.

In [ ]:
# Per-frequency air-bone gap
print('Left ABG:', left.abg())
print('Right ABG:', right.abg())

# ABG PTA — default 500/1000/2000/4000
print('\nLeft ABG PTA (WHO):', left.abg_pta())
print('Right ABG PTA (WHO):', right.abg_pta())

# AAO-HNS method — 500/1000/2000/3000 with fallback
print('Left ABG PTA (AAO-HNS):', left.abg_pta(standard='aao_hns'))

# Loss type classification
print('\nLeft loss type:', left.loss_type())
print('Right loss type:', right.loss_type())

In [ ]:
# mask_warning — flags unmasked bone thresholds
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    # Build an ear with some unmasked bone
    ear_unmasked = EarAudiogram(
        air={500: 40, 1000: 45, 2000: 50},
        bone={500: ThresholdPoint(10.0), 1000: ThresholdPoint(15.0, masked=True), 2000: ThresholdPoint(20.0)},
    )
    ear_unmasked.abg(mask_warning=True)
    if w:
        print('Warning:', str(w[0].message))
    else:
        print('No warning')

## BinauralAudiogram

Both ears + test-level metadata.

In [7]:
ba = BinauralAudiogram(
    left=left,
    right=right,
    audiogram_id='aud-2024-001',
    subject_id='pt-042',
    performed_at='2024-06-15',
    source='clinic',
)

# Shortcuts
print('L is left:', ba.L is ba.left)
print('R is right:', ba.R is ba.right)

# Direct threshold access
print('Right air 4000:', ba.get_threshold(4000, 'right'))
print('Left bone 1000:', ba.get_threshold(1000, 'left', pathway='bone'))

L is left: True
R is right: True
Right air 4000: ThresholdPoint(threshold_db=120.0, masked=False, nr=True)
Left bone 1000: ThresholdPoint(threshold_db=20.0, masked=True, nr=False)


In [8]:
# Binaural PTA
print('PTA both ears:', ba.pta())
print('Better ear PTA:', ba.better_ear_pta())
print('Worse ear PTA:', ba.worse_ear_pta())

# Symmetry (left - right per frequency)
print('\nSymmetry (left - right):')
pprint(ba.symmetry())

PTA both ears: {'left': 28.333333333333332, 'right': 51.666666666666664}
Better ear PTA: 28.333333333333332
Worse ear PTA: 51.666666666666664

Symmetry (left - right):
{250: -5.0, 500: -25.0, 1000: -25.0, 2000: -20.0, 4000: -70.0, 8000: -55.0}


## Asymmetry

Named criteria or custom callable.

In [9]:
print('Available criteria:', sorted(ASYMMETRY_CRITERIA))
print()

for name in ASYMMETRY_CRITERIA:
    print(f'  {name}: {ba.is_asymmetric(name)}')

Available criteria: ['any_15db', 'nr_one_side', 'pta_15db', 'two_consecutive_10db']

  any_15db: True
  two_consecutive_10db: True
  pta_15db: True
  nr_one_side: True


In [10]:
# Custom criterion — lambda or function
custom = lambda ba: abs(ba.pta()['left'] - ba.pta()['right']) > 20
print('Custom (PTA diff > 20 dB):', ba.is_asymmetric(custom))

# Interaural differences
print()
for d in ba.interaural_differences():
    flag = ' ← NR involved' if d.nr_involved else ''
    print(f'  {d.freq_hz:5d} Hz  diff={d.difference_db:+.0f} dB  better={d.better_ear}{flag}')

Custom (PTA diff > 20 dB): True

    250 Hz  diff=-5 dB  better=left
    500 Hz  diff=-25 dB  better=left
   1000 Hz  diff=-25 dB  better=left
   2000 Hz  diff=-20 dB  better=left
   4000 Hz  diff=-70 dB  better=left ← NR involved
   8000 Hz  diff=-55 dB  better=left ← NR involved


## Summary

`summary()` returns a flat dict of all computed metrics — PTA, severity, ABG, loss type,
asymmetry, speech, and frequency counts.

Pass `standard="aao_hns"` to use AAO-HNS method for severity, ABG, and loss type.

In [ ]:
# Full summary — WHO 2021 default
print('=== Summary (WHO 2021) ===')
for k, v in sorted(ba.summary().items()):
    print(f'  {k}: {v}')

print()

# AAO-HNS — flows through to severity, ABG, and loss type
print('=== Summary (AAO-HNS) ===')
for k, v in sorted(ba.summary(standard='aao_hns').items()):
    print(f'  {k}: {v}')

In [ ]:
# Filter to specific categories
print('ABG + severity only:')
pprint(ba.summary(include=['abg', 'severity'], standard='aao_hns'))